# HISTORICAL CUSTOMER LIFETIME VALUE

# Step 1: Import libraries and load dataset

In [ ]:
import pandas as pd
import numpy as np
import os
print("Libraries imported succesfully")

df = pd.read_csv(r"C:\Users\onyer\Downloads\cleaned_retail_data_updated\cleaned_retail_data_updated.csv")
print(f"\nDataset shape: {df.shape}")
print(f"\nColums: {df.columns.tolist()}")
df.head()

# Step 2: Calculate TotalRevenue per transaction

In [ ]:
df["TotalRevenue"] = df["Quantity"] * df["UnitPrice"]

print("TotalRevenue Column added successfully")
print(df[["CustomerID", "InvoiceNo", "Quantity", "UnitPrice", "TotalRevenue"]])

# Step 3: AOV

In [ ]:
try:
    aov = pd.read_csv('outputs/siva_aov.csv')
    print("✅ siva AOV output loaded successfully")

except FileNotFoundError:
    print("⚠️ siva output not found - using fallback AOV calculation")
    
    customer_revenue = df.groupby('CustomerID')['TotalRevenue'].sum()
    customer_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()
    aov = (customer_revenue / customer_orders).reset_index()
    aov.columns = ['CustomerID', 'AOV']

print(aov.head())

# Step 4: Purchase frequency

In [ ]:
try:
    purchase_freq = pd.read_csv('outputs/yash_frequency.csv')
    print("✅ yash output loaded successfully")

except FileNotFoundError:
    print("⚠️ yash output not found - using fallback")
    
    purchase_freq = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
    purchase_freq.columns = ['CustomerID', 'PurchaseFrequency']

print(purchase_freq.head())

# Step 5: Historical CLTV

In [ ]:
# Merge AOV and Purchase Frequency
cltv_df = aov.merge(purchase_freq, on='CustomerID')

# CLTV = AOV x Purchase Frequency
cltv_df['CLTV'] = cltv_df['AOV'] * cltv_df['PurchaseFrequency']

print(cltv_df.head(10))
print(f"\nCLTV Summary:\n{cltv_df['CLTV'].describe()}")

In [ ]:
os.makedirs('outputs', exist_ok=True)
cltv_df.to_csv('outputs/hiren_cltv.csv', index=False)
print("Saved successfully")

# Step 6: Project 12-Month CLTV

In [ ]:
# First merge CohortMonth into cltv_df
cltv_df = cltv_df.merge(df[['CustomerID', 'CohortMonth']].drop_duplicates(), 
                          on='CustomerID', how='left')

# Project 12-Month CLTV per customer
cltv_df['CLTV_12Month'] = cltv_df['CLTV'] * 12

# Group by CohortMonth segment
cohort_cltv = cltv_df.groupby('CohortMonth').agg(
    Total_Customers=('CustomerID', 'count'),
    Avg_AOV=('AOV', 'mean'),
    Avg_Purchase_Frequency=('PurchaseFrequency', 'mean'),
    Avg_CLTV=('CLTV', 'mean'),
    Avg_12Month_CLTV=('CLTV_12Month', 'mean'),
    Total_12Month_CLTV=('CLTV_12Month', 'sum')
).round(2).reset_index()

print("12-Month CLTV Per Cohort Segment:")
print(cohort_cltv)
print(f"\nBest Performing Cohort:")
print(cohort_cltv.loc[cohort_cltv['Avg_12Month_CLTV'].idxmax()])
print(f"\nWorst Performing Cohort:")
print(cohort_cltv.loc[cohort_cltv['Avg_12Month_CLTV'].idxmin()])